In [ ]:
import numpy as np
import pandas as pd
import time
import zipfile
import glob
import os
from urllib.parse import urlparse
import re
import warnings

warnings.filterwarnings('ignore')
def CodeAnalyzer(WEBs):
    title,NODES = None,{}
    equal,qmark,amp,digit,letter,specialchar = 0,0,0,0,0,0
    lineofcode,longestlinelen,selfhref,emptyref,extref,img,css,js,popup,iframe = 0,0,0,0,0,0,0,0,0,0
    
    #PATH = open(FILE, 'r', encoding="utf8")
    #FILEDATA = PATH.read()
    LINES = WEBs.decode('utf-8').split('\n')
    fLine=LINES[0].lower()
    NODES['FILENAME'] = fLine[0:fLine.rfind(' ')] + '.txt'
    URL = fLine[fLine.find(' ')+1:len(fLine)]
    #print(URL)
    NODES['url'] = URL
    urlLen = len(URL)
    NODES['url_len'] = urlLen
    domain = urlparse(URL).netloc
    TLD = domain[domain.rfind('.')+1:len(domain)]
    NODES['domain'] = domain
    NODES['url_domainlen'] = len(domain)
    #a domain can be only ip without port number. Hence checking port number also in domain name may not be accurate in all cases
    pattern = re.compile("[0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3}\.[0-9]{1,3}")
    NODES['isdomainanip'] = 0
    if(pattern.match(domain)!=None):
        NODES['isdomainanip'] = 1
    NODES['tld'] = TLD
    NODES['tldlen'] = len(TLD)
    URL = URL.replace("https://","").replace("http://","").replace("www.","")
    for ch in URL:
        if ch.isdigit():
            digit=digit+1  
        elif ch.isalpha():
            letter=letter+1
        elif(ch=='='):
            equal=equal+1
        elif(ch=='?'):
            qmark=qmark+1
        elif(ch=='%'):
            amp=amp+1
        else:
            specialchar=specialchar+1
    
    
    sldomain = domain[domain.rfind(".",0,domain.rfind("."))+1:len(domain)]
    NODES['url_noofsubdomains'] = domain.count('.')-1
    #char obfuscation is used in url using % followed with any 2 combination of digit or char between A-F
    NODES['url_hasobfuscation'] = 0
    NODES['url_noofobfuscatedchar'] = 0
    NODES['url_obfuscationratio'] = 0
    match  =  len(re.findall(r"%[A-F,0-9][A-F,0-9]",URL))
    if(match>0):
        NODES['url_hasobfuscation'] = 1
        NODES['url_noofobfuscatedchar'] = match*3
        NODES['url_obfuscationratio'] = str(round((match*3)/urlLen, 3))
    NODES['url_letter'] = letter
    NODES['url_letter_ration'] = str(round(letter/urlLen, 3))
    NODES['url_digit'] = digit
    NODES['url_digit_ration'] = str(round(digit/urlLen, 3))
    NODES['url_equal'] = equal
    NODES['url_qmark'] = qmark
    NODES['url_amp'] = amp
    NODES['url_otherspecialchar'] = specialchar  
    NODES['url_spclchar_ration'] = str(round((equal+qmark+amp+specialchar)/urlLen, 3))
    
    if(fLine.find('https')>=0):
        NODES['url_ssl'] = 1
    else:
        NODES['url_ssl'] = 0
    NODES['unreachable'],NODES['lineofcode'], NODES['largestlinelen'], NODES['hastitle'] = 0,0,0,0
    NODES['title'],NODES['titlematchscore'],NODES['favicon'],NODES['robots'],NODES['responsive'],NODES['urlredirect'],NODES['selfredirect'],NODES['description'] = 0,0,0,0,0,0,0,0
    NODES['popup'],NODES['iframe'],NODES['formsubmitexternal'] = 0,0,0
    NODES['socialnet'],NODES['submitbutton'],NODES['hiddenfield'],NODES['passwordfield'] = 0,0,0,0,0
    NODES['bank'],NODES['pay'],NODES['crypto'], NODES['hascopyrightinfo']  = 0,0,0,0
    
    for LINE in LINES:
        try:
            LINE = LINE.lower()
            if(len(LINE)>longestlinelen):
                longestlinelen=len(LINE)
            lineofcode = lineofcode+1
            LINE=LINE.replace(' ','').lower()
            if(LINE.find('redirectingto')>=0):
                NODES['urlredirect']=1
            if(LINE.find('forbidden')>=0 or LINE.find('accessdenied')>=0 or LINE.find('invalidurl')>=0 or LINE.find('servererror')>=0 or LINE.find('humanverification')>=0  or LINE.find('movedpermanently')>=0):
                NODES['unreachable']=1
            if(LINE.find('href')>=0):
                if(LINE.find('http')>=0):
                    extref=extref+1
                elif(LINE.find('="#"')>=0):
                    emptyref=emptyref+1
                if(LINE.find(domain)>=0 or LINE.find('="/')>=0):
                    selfhref=selfhref+1
            if(LINE.find('.jpg')>=0 or LINE.find('.jpeg')>=0 or LINE.find('.png')>=0):
                img=img+1 
            if(LINE.find('.css')>=0):
                css=css+1          
            if(LINE.find('.js')>=0):
                js=js+1
            if(LINE.find('window.open')>=0):
                popup=popup+1
            if(LINE.find('iframe')>=0):
                iframe=iframe+1
            if(LINE.find('<title>')>=0 or LINE.find('</title>')>=0):
                #NODES['title'] = LINE[LINE.find('<title>')+7:LINE.find('</title>')]
                NODES['hastitle']=1 
            if(LINE.find('linkrel')>=0 and LINE.find('favicon')>=0):
                NODES['favicon']=1
            if(LINE.find('metaname')>=0):
                if(LINE.find('robots')>=0):
                    NODES['robots']=1
                elif(LINE.find('viewport')>=0):
                    NODES['responsive']=1
                elif(LINE.find('description')>=0):
                    NODES['description']=1    
            if((LINE.find('http-equiv')>=0 and LINE.find('refresh')>=0) or (LINE.find('window.location=')>=0)
                 or (LINE.find('window.location.replace')>=0) or (LINE.find('window.location.href=')>=0) 
                or (LINE.find('http.open')>=0) ):
                NODES['urlredirect']=1
                if(LINE.find(sldomain)>=0): #if redirecting to own domain, case of benign url
                    NODES['selfredirect']=1
            if(LINE.find('www.facebook.com')>=0 or LINE.find('www.linkedin')>=0 or LINE.find('www.youtube')>=0 or LINE.find('www.twitter.com')>=0):  
                NODES['socialnet']=1
            if(LINE.find('bank')>=0 or LINE.find('wallet')>=0):
                NODES['bank']=1
            if(LINE.find('pay')>=0):
                NODES['pay']=1
            if(LINE.find('crypto')>=0):
                NODES['crypto']=1
            if(LINE.find('type')>=0):
                if(LINE.find('submit')>=0):
                    NODES['submitbutton']=1
                elif(LINE.find('hidden')>=0):
                    NODES['hiddenfield']=1
                elif(LINE.find('password')>=0):
                    NODES['passwordfield']=1

            if(LINE.find('formclass')>=0 and LINE.find('method')>=0 and LINE.find('action="/')<0 ):
                NODES['formsubmitexternal']=1 
            if(LINE.find('&copy;')>=0 or LINE.find('allright')>=0 or LINE.find('copyright')>=0 or LINE.find('©')>=0 or LINE.find('rightsreserved')>=0):
                NODES['hascopyrightinfo']=1
                      
        except Exception as ex:
            print(ex)
                   
    #print(URL)
    #print(domain)
    #print(NODES)
    NODES['lineofcode'] = lineofcode
    NODES['largestlinelen'] = longestlinelen
    
    NODES['noofimg'] = img
    NODES['noofcss'] = css
    NODES['noofjs'] = js
    NODES['noofselfhref']=selfhref
    NODES['noofemptyref'] = emptyref
    NODES['noofextref'] = extref
    NODES['popup'] = popup
    NODES['iframe'] = iframe
    
    WEBs = str(WEBs)
    WEBs = WEBs.lower()
    if(NODES['hastitle']==1):
        strt = WEBs.find('<title>')
        last = WEBs.find('</title>')
        title = WEBs[strt+7:last].replace("\\n", "").replace("&nbsp;", "")
        tiny = URL.replace("https://","").replace("http://","").replace("www.","").replace(TLD,"").replace(".","").replace("/","")
        tSet = set(title.split())
        score = 0
        for element in tSet:
            n = len(element)
            for i in range(n,1,-1):
                sub = element[0:i]
                if(tiny.find(sub)>=0):
                    score = score + len(sub)
                    break
        score = (score/len(tiny))*100
        NODES['title'] = tiny
        NODES['titlematchscore'] = score

    # 1 for bebign websites
    NODES['label'] = 0
    df = pd.DataFrame(NODES, index=[0])
    return df

In [ ]:
#"D:\\Backup_Work\\P7\\CODE\\SITES\Phishing\\*.*"):
MASTER = pd.DataFrame()
txtfiles = []
count = 0

ZIPs = zipfile.ZipFile("C:\\Users\\Arvind\dataset\\P7Final7z\\PhishingSet6_Finished.zip", "r")
tot = len(ZIPs.namelist())
for WEB in ZIPs.namelist():
    try:
        webData = ZIPs.read(WEB)
        df = CodeAnalyzer(webData)
        count = count + 1  
        if(count==1):
            MASTER=df
            AddCols = df.columns
        else:
            OlsCols = MASTER.columns
            NewCols = df.columns
            AddCols = NewCols.drop(OlsCols, errors = 'ignore')
            for COL in AddCols:
                MASTER[COL]=0
            MASTER = MASTER.append(df)
        if(count%1000==0):
            print(tot-count)
    except Exception as ex:
        print(tot-count, " ", WEB, " : ", ex)
MASTER = MASTER.replace(np.nan,0)
MASTER = MASTER.reset_index(drop=True)
print("\nTotal Columns : ", len(MASTER.columns))

COLS = MASTER.columns
for CL in COLS:
    try:
        MASTER[CL] = MASTER[CL].apply(np.int64)
    except:
        pass

MASTER.to_csv(r'D:\P7Code\Reports\FinalPhishingSet66.csv',index = False)
MASTER

In [1]:
fLine='116886 https://www.winnebago.com'
nm = fLine[0:fLine.rfind(' ')] + '.txt'
URL = fLine[fLine.find(' ')+1:len(fLine)]
print(nm)
print(URL)

116886.txt
https://www.winnebago.com
